# **Mapping short-let pressure in Bristol: A neighbourhood analysis of Airbnb listings**

### Objectives
This project analyses Inside Airbnb listing data for Bristol to help local authority policy and planning teams understand where professional short-let operators are concentrated, and how their listings differ from those of casual hosts.

A professional operator is defined as a host with three or more listings in the data. The threshold is a judgement call, so findings are re-tested at 2, 5 and 10 listings.

The analysis tests three hypotheses (area concentration, listing behaviour, ratings) and trains a classifier (decision tree compared with logistic regression) to show which listing characteristics separate operator listings from casual ones. An interactive dashboard lets users explore the findings by neighbourhood.

The data measures short-let supply. It cannot show housing pressure, displacement or harm, and no causal claims are made.

This is an independent portfolio project written for an imagined council audience. It is not commissioned by or affiliated with any local authority.

### Project Plan
This project follows a structured analytics workflow. Each stage is documented in the notebook with the reasoning behind decisions, not just the code.

- Define the problem. Set the question, the audience (an imagined local authority policy team), the definition of an operator, and the hypotheses.
- Extract. Load the Inside Airbnb Bristol snapshot (listings, neighbourhood lookup and boundary GeoJSON), keeping the raw files unmodified.
- Check the reference data. Confirm that neighbourhood names are consistent across all three files so they can be joined safely.
- Investigate the listings data. Inspect structure, data types, missing values, constant columns and distributions.
- Clean and transform. Flag dormant listings, drop empty, constant and personal-data columns (each with a recorded reason), convert data types, handle missing values, check outliers, create the operator and other features, then validate and save.
- Build the neighbourhood-level table. Aggregate listings into one row per neighbourhood (34 rows) with the concentration measures.
- Exploratory data analysis and hypothesis tests. Describe distributions, then test H1 (chi-square), H2 (Mann-Whitney with effect sizes) and H3.
- Machine learning. Train and compare a decision tree and a logistic regression to predict operator status, excluding the host-count columns as inputs, and justify the choice.
- Interpret the results. Explain which characteristics separate operators from casual hosts, and state the limitations.
- Build an interactive dashboard in [DECIDE: Power BI, Tableau or Streamlit].
- Document and reflect. Complete the README, record limitations and alternative approaches, and reflect on what was learned.

### Inputs

Data: 

Inside Airbnb, Bristol snapshot dated [DECIDE: 22 or 25 September 2026], downloaded on [date]. Licence: [copy the exact wording from insideairbnb.com/get-the-data]. https://insideairbnb.com/get-the-data/

The files used:

- listings.csv.gz: detailed listings, the main table for measuring short-let activity
- neighbourhoods.csv: neighbourhood names, the unit of analysis
- neighbourhoods.geojson: neighbourhood boundaries, for map visualisations later

Outputs:

- data/processed/[file name]: the cleaned listings table (named once step 8 is done)
- docs/dropped_columns.csv: every dropped column with its reason
- Neighbourhood-level table, test results, trained models and dashboard, added as those stages are completed

Additional Comments:

Hosts are pseudonymised with an anonymous host_key. Listing IDs and the raw files still identify hosts, so [DECIDE: state here whether data/raw is committed to the public repository].

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol'

# Defining the problem 

## Problem Statement and Hypotheses

### Context
Short-term lets (such as Airbnb listings) range from residents occasionally renting a spare room to businesses running many whole homes as full-time lettings. These have different implications for housing supply, neighbourhoods and regulation, but a listing platform does not distinguish between them. Local authorities have limited visibility of which type of activity is happening where. 

### Core question
**Where in Bristol are professional short-let operators concentrated, and how do their listings differ from those of casual hosts?**

### Definitions
- **Professional operator listing:** a listing whose host has **3 or more listings** in the data. This threshold is a judgement call, so the analysis will be repeated at other thresholds (for example 2, 5 and 10) to check how sensitive the findings are. The data dictionary [link] says that calculated_host_listings_count concerns is a measure of "The number of listings the host has in the current scrape, in the city/region geography"
- **Casual host listing:** a host with 1 or 2 listings.
- **Concentration:** the share of an area's listings that are operator listings.

### Hypotheses
**H1: Concentration varies by area.** The share of operator listings is not the same across Bristol's 34 neighbourhoods. (Null: it does not differ beyond chance.)

**H2: Operator listings behave differently.** Compared with casual hosts, operator listings have higher annual availability and different minimum-stay requirements, consistent with full-time commercial lettings. (Null: no difference.)

**H3: Operator listings are rated differently.** Review scores differ between operator and casual listings. An exploratory check suggested operator hosts score lower, but this needs testing properly on cleaned data. (Null: no difference.)

### Role of machine learning
A classifier (decision tree compared with logistic regression) will be trained to predict whether a listing is an operator listing, using observable listing characteristics such as room type, availability, minimum stay, price, capacity and area. **The host-listing-count columns are excluded from the inputs**, otherwise the model would simply read the answer. The aim is not prediction for its own sake. It is to identify *which characteristics best separate operator listings from casual ones*, which gives the council interpretable evidence.

### Why this matters to a local authority
- **Targeting finite resources.** Policy and enforcement capacity are limited. Knowing where commercial-style activity is concentrated, rather than treating all short-lets alike, helps direct attention where it is most relevant.
- **Proportionate responses.** A resident occasionally renting a room and an operator running a portfolio of whole homes raise different questions. Evidence that separates them supports responses that fit each.
- **Evidence where data is thin.** The `license` field is empty for every Bristol listing in this dataset, so the platform data alone offers no registration information. An open-data analysis gives an independent starting point. [Add the current registration or licensing position in England, with a source.]
- **Consumer and neighbourhood quality.** If operator listings are rated differently, that may bear on visitor experience and complaints. [Add a source if you keep this point.]

### Scope and limitations
- This analysis measures **supply**, not housing pressure. It cannot show displacement, rent effects or harm to residents, and no causal claims will be made.
- The data is a single snapshot (22 September 2026) of listings active at that time. It does not show change over time or delisted properties.
- About a quarter of listings (705 of 2,932) are marked in the data as coming from a previous scrape, not the latest one. Their availability and price may be older or missing, so key results are checked with and without them.
- Hosts are identified only within this dataset. A host may have listings elsewhere.
- Prices are asking prices, and 23% of listings have none (mostly listings with no availability).
- Some areas have few listings (17 at the lowest), so area-level percentages there are unreliable.
- A statistical difference between groups does not show that operators *cause* a difference.

# Importing the data 

### Method
- Raw files are stored unmodified in data/raw/. All cleaning happens on copies, so the original download can always be recovered.
- listings.csv.gz is read directly by pandas, which decompresses it automatically.
- The GeoJSON is loaded with Python's json module as a dictionary. At this stage it is only needed to check area names and, later, to feed the dashboard map, so a mapping library such as geopandas would add a dependency without adding anything the checks need. [DECIDE: keep or revise this reason.]
- Paths are relative to the project root, so the notebook runs on other machines without editing.



In [4]:
from pathlib import Path
import pandas as pd
import json

In [5]:
DATA_RAW = Path(r"c:\Users\chheb\OneDrive\Documents\Programming\Mapping_Short_Let-Pressure_in_Bristol\Data\raw")  


listings = pd.read_csv(DATA_RAW / "listings.csv.gz")        
neighbourhoods = pd.read_csv(DATA_RAW / "neighbourhoods.csv")

with open(DATA_RAW / "neighbourhoods.geojson", encoding="utf-8") as f:
    neighbourhoods_geo = json.load(f)  

### Confirming the import worked
- The data folder path was checked with `DATA_RAW.exists()`, which returned `True`.
- `listings.shape` returned **(2,932, 90)**: 2,932 listings and 90 columns.


In [6]:
print(DATA_RAW.exists())


True


In [7]:
print(listings.shape)

(2932, 90)


In [8]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,70820,https://www.airbnb.com/rooms/70820,20260922161704,2026-09-25,NaN,City View - Sarah's double room.,A good sized room with a comfy double bed and ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,360195,...,4.93,4.76,4.76,NaN,NaN,5,0,5,0,1.36
1,176306,https://www.airbnb.com/rooms/176306,20260922161704,2026-09-25,NaN,The White Room - Central Bristol Art House Ga...,"A small double room in an artist's house, with...",NaN,https://a0.muscache.com/pictures/e1e7fdfc-9db5...,708175,...,4.90,4.78,4.69,NaN,NaN,1,0,1,0,0.36
2,307634,https://www.airbnb.com/rooms/307634,20260922161704,2026-09-25,NaN,"'The Sunset Room' Peaceful, Clear Home","Close to Bristol and Bath Railway Path, leadin...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,1582810,...,4.84,4.86,4.74,NaN,NaN,3,0,3,0,0.47
3,390781,https://www.airbnb.com/rooms/390781,20260922161704,2026-09-25,previous scrape,"Cosy large room nr TV studios, Paintworks & st...",A room with a shared bathroom in an old Victor...,NaN,https://a0.muscache.com/pictures/9755283/2e0bb...,1954565,...,4.93,4.72,4.84,NaN,NaN,2,0,2,0,0.34
4,407485,https://www.airbnb.com/rooms/407485,20260922161704,2026-09-25,NaN,Bristol Beautiful Garden Apartment,"<b>The space</b><br />This is a sunny, light h...",NaN,https://a0.muscache.com/pictures/58023936/aa2b...,2029719,...,4.92,4.79,4.70,NaN,NaN,1,1,0,0,2.35


### First observations
- 2,932 listings is a modest sample for a whole city. Some neighbourhoods have very few listings (17 in the smallest), so area-level percentages need care and the area tests must account for small counts.
- 90 columns is more than the analysis needs. 15 are entirely empty, and several have the wrong type (for example price is text with a $). A later step selects the relevant columns and records the reason for each drop.
- Most of the structure is clear from info() and head(), but whether a listing is genuinely active is not. That is handled in the transformation stage.

---

# Exploratory data analysis

## 1. Lookup files


`neighbourhoods.csv` and `neighbourhoods.geojson` are lookup files, not analytical data. They supply the area names (and, for the GeoJSON, the boundary shapes) that listings are grouped by. The aim is to confirm they can be joined to the listings safely, not to explore distributions.

**Steps**
1. Checked the CSV's structure with `.shape`, `.info()` and `.head()`.
2. Counted nulls per column and duplicates in the name column.
3. Checked for stray whitespace in names, since an invisible trailing space would make two identical-looking names fail to match.
4. Compared the area names in `listings["neighbourhood_cleansed"]` against the lookup table using set differences, in both directions.
5. Inspected the GeoJSON structure (top-level keys, number of features, and the properties of the first feature) to find which key holds the area name.
6. Compared the GeoJSON area names against the lookup table, in both directions.


In [9]:
neighbourhoods.shape

(34, 2)

In [10]:
neighbourhoods.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   neighbourhood_group  0 non-null      float64
 1   neighbourhood        34 non-null     str    
dtypes: float64(1), str(1)
memory usage: 1.1 KB


In [11]:
neighbourhoods.head()

,neighbourhood_group,neighbourhood
0,NaN,Ashley
1,NaN,Avonmouth & Lawrence Weston
2,NaN,Bedminster
3,NaN,Bishopston & Ashley Down
4,NaN,Bishopsworth


In [12]:
print(neighbourhoods.isnull().sum())
print("Duplicated names:", neighbourhoods["neighbourhood"].duplicated().sum())

neighbourhood_group    34
neighbourhood           0
dtype: int64
Duplicated names: 0


In [13]:
names = neighbourhoods["neighbourhood"]
print("Names with stray whitespace:", (names != names.str.strip()).sum())

Names with stray whitespace: 0


In [14]:
print("Null areas in listings:", listings["neighbourhood_cleansed"].isnull().sum())

listing_areas = set(listings["neighbourhood_cleansed"].dropna().unique())
lookup_areas = set(neighbourhoods["neighbourhood"])

print("In listings but not in lookup:", listing_areas - lookup_areas)
print("In lookup but not in listings:", lookup_areas - listing_areas)

Null areas in listings: 0
In listings but not in lookup: set()
In lookup but not in listings: set()


In [15]:
area_counts = listings["neighbourhood_cleansed"].value_counts()
print(area_counts.describe())
print("Smallest areas:\n", area_counts.tail(3))
print("Largest areas:\n", area_counts.head(3))


neighbourhoods = neighbourhoods.drop(columns="neighbourhood_group")
print(neighbourhoods.shape) 

count     34.000000
mean      86.235294
std       68.863527
min       17.000000
25%       38.250000
50%       67.000000
75%      114.750000
max      314.000000
Name: count, dtype: float64
Smallest areas:
 neighbourhood_cleansed
Henbury & Brentry         23
Hartcliffe & Withywood    18
Stockwood                 17
Name: count, dtype: int64
Largest areas:
 neighbourhood_cleansed
Ashley     314
Central    254
Clifton    205
Name: count, dtype: int64
(34, 1)


**Findings: CSV lookup**
- The table has **34 areas** and 2 columns. `.head()` shows area names (for example Ashley, Avonmouth & Lawrence Weston, Bedminster) alongside a blank `neighbourhood_group`.
- There are no duplicated names and no whitespace issues.
- `neighbourhood_group` is **empty for all 34 rows**, which is why pandas read it as `float64`. It holds no information for Bristol.

**Findings: listings against lookup**
- All 2,932 listings have an area (0 nulls).
- The set of areas in `listings` is **identical** to the set in the lookup table. No listing is orphaned and no area is empty.

In [16]:
print(neighbourhoods_geo.keys())
print("Number of features:", len(neighbourhoods_geo["features"]))
print(neighbourhoods_geo["features"][0]["properties"]) 

dict_keys(['type', 'features'])
Number of features: 34
{'neighbourhood': 'Windmill Hill', 'neighbourhood_group': None}


In [17]:
geo_areas = {f["properties"]["neighbourhood"] for f in neighbourhoods_geo["features"]}

print("In GeoJSON but not in lookup:", geo_areas - lookup_areas)
print("In lookup but not in GeoJSON:", lookup_areas - geo_areas)

In GeoJSON but not in lookup: set()
In lookup but not in GeoJSON: set()


**Findings: GeoJSON**
- Top-level keys are `type` and `features`, with **34 features**, one per area. This matches the CSV row count.
- Each feature's `properties` holds the area name under the key `neighbourhood` (plus an empty `neighbourhood_group`, consistent with the CSV).
- GeoJSON names match the lookup table **exactly in both directions**: no names appear in one but not the other.

**Decisions**
- **Drop `neighbourhood_group`.** It contains no values, so it cannot support any analysis.
- **No renaming or text cleaning needed.** Names match exactly across all three files, so I can join on `neighbourhood_cleansed` and use the GeoJSON for maps without a mapping step.
- **Use `neighbourhood_cleansed` as the area key.** The raw `neighbourhood` column in the listings file is entirely empty.
- **Leave the boundary shapes untouched for now.** They are only needed for the dashboard map.

**Limitations and open questions**
- These checks show that names are consistent across files, not that the boundaries suit this analysis.  Any external data I join later (for example, household counts) will need to use the same boundaries.
- Areas are very uneven in size, from [smallest] to [largest] listings. This affects how reliable any area-level percentage will be, and is addressed when the neighbourhood-level table is built.

## 2. 'listings': analytical data

### A. Structure and data types

In [18]:
listings.describe()

,id,scrape_id,neighborhood_overview,host_id,host_profile_id,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,2.932000e+03,2.932000e+03,0.0,2.932000e+03,2.932000e+03,0.0,2932.000000,2932.000000,2932.000000,2932.000000,...,2644.000000,2644.000000,2644.000000,0.0,0.0,2932.000000,2932.00000,2932.000000,2932.000000,2644.000000
mean,9.072239e+17,2.026092e+13,NaN,3.503762e+16,1.481828e+18,NaN,7.926330,5.256139,5.155866,5.189632,...,4.851880,4.767613,4.691566,NaN,NaN,12.156207,11.28206,0.827763,0.011937,1.692663
std,6.413870e+17,0.000000e+00,NaN,2.424918e+17,5.380515e+16,NaN,3.822936,3.316709,3.671786,3.316705,...,0.312867,0.283084,0.393268,NaN,NaN,28.026407,27.64082,1.520924,0.244057,1.886890
min,7.082000e+04,2.026092e+13,NaN,2.381000e+03,1.462506e+18,NaN,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,NaN,NaN,1.000000,0.00000,0.000000,0.000000,0.010000
25%,5.134202e+07,2.026092e+13,NaN,3.759181e+07,1.463080e+18,NaN,4.000000,2.750000,2.000000,2.000000,...,4.830000,4.680000,4.600000,NaN,NaN,1.000000,1.00000,0.000000,0.000000,0.400000
50%,1.055818e+18,2.026092e+13,NaN,1.226961e+08,1.467231e+18,NaN,9.000000,5.000000,4.000000,5.000000,...,4.950000,4.840000,4.800000,NaN,NaN,2.000000,1.00000,0.000000,0.000000,1.010000
75%,1.479047e+18,2.026092e+13,NaN,4.267019e+08,1.469991e+18,NaN,11.000000,8.000000,9.000000,8.000000,...,5.000000,4.950000,4.910000,NaN,NaN,6.000000,4.00000,1.000000,0.000000,2.380000
max,1.779673e+18,2.026092e+13,NaN,1.775395e+18,1.775395e+18,NaN,18.000000,11.000000,15.000000,11.000000,...,5.000000,5.000000,5.000000,NaN,NaN,129.000000,126.00000,11.000000,5.000000,14.520000


In [19]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 2932 entries, 0 to 2931
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            2932 non-null   int64  
 1   listing_url                                   2932 non-null   str    
 2   scrape_id                                     2932 non-null   int64  
 3   last_scraped                                  2932 non-null   str    
 4   source                                        705 non-null    str    
 5   name                                          2932 non-null   str    
 6   description                                   2924 non-null   str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   2932 non-null   str    
 9   host_id                                       2932 non-null   int64  
 10 

In [20]:
listings.dtypes.value_counts()
listings.select_dtypes("object").columns.tolist() 

C:\Users\chheb\AppData\Local\Temp\ipykernel_9304\2973873790.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  listings.select_dtypes("object").columns.tolist()


['listing_url',
 'last_scraped',
 'source',
 'name',
 'description',
 'picture_url',
 'host_url',
 'host_profile_url',
 'host_name',
 'host_location',
 'host_about',
 'host_is_superhost',
 'host_picture_url',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood_cleansed',
 'property_type',
 'room_type',
 'bathrooms_text',
 'amenities',
 'price',
 'price_quote_checkin_date',
 'price_quote_checkout_date',
 'price_quote_raw',
 'has_availability',
 'calendar_last_scraped',
 'first_review',
 'last_review']

In [21]:
listings[["price", "first_review", "last_review"]].head(10)

,price,first_review,last_review
0,$52.33,2013-10-18,2026-07-29
1,$58.00,2012-11-18,2025-12-02
2,$53.50,2017-06-04,2026-09-19
3,NaN,2012-03-26,2026-04-23
4,$134.00,2012-04-30,2026-09-06
5,$77.00,2012-07-23,2026-09-17
6,$272.00,2012-04-30,2026-09-13
7,$170.00,2012-11-15,2026-09-17
8,NaN,2018-03-29,2020-03-01
9,$189.00,2013-07-09,2026-09-13


### B. Missing values

In [22]:
missing = listings.isnull().sum().to_frame("missing")
missing["percent"] = (missing["missing"] / len(listings) * 100).round(1)
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)

missing

,missing,percent
host_since,2932,100.0
neighborhood_overview,2932,100.0
host_thumbnail_url,2932,100.0
host_acceptance_rate,2932,100.0
host_response_rate,2932,100.0
host_response_time,2932,100.0
host_neighbourhood,2932,100.0
host_total_listings_count,2932,100.0
host_verifications,2932,100.0
neighbourhood,2932,100.0


### C. Empty columns

In [23]:
constant_cols = [c for c in listings.columns if listings[c].nunique(dropna=False) <= 1]
print(len(constant_cols), constant_cols)

18 ['scrape_id', 'last_scraped', 'neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated', 'calendar_last_scraped', 'license', 'instant_bookable']


### D. Duplicates and identifiers

In [24]:
print("Duplicate rows:", listings.duplicated().sum())
print("IDs unique:", listings["id"].is_unique)

Duplicate rows: 0
IDs unique: True


### E. Distributions

Summary statistics hide skew. Comparing the mean with the median, and the median with the 99th percentile, shows which columns have extreme values that could distort averages. This also sets up the outlier check in the transformation stage.

In [25]:
print(listings["room_type"].value_counts(normalize=True).round(3))

key_cols = ["accommodates", "minimum_nights", "availability_365",
            "number_of_reviews", "calculated_host_listings_count"]
dist = listings[key_cols].describe(percentiles=[0.5, 0.9, 0.99]).T
dist["mean_minus_median"] = (dist["mean"] - dist["50%"]).round(1)
dist

room_type
Entire home/apt    0.721
Private room       0.275
Hotel room         0.002
Shared room        0.002
Name: proportion, dtype: float64


,count,mean,std,min,50%,90%,99%,max,mean_minus_median
accommodates,2932.0,3.685880,2.488301,1.0,3.0,7.0,12.00,16.0,0.7
minimum_nights,2931.0,3.611737,13.755547,1.0,2.0,5.0,30.00,365.0,1.6
availability_365,2932.0,176.613233,141.545401,0.0,173.0,356.0,365.00,365.0,3.6
number_of_reviews,2932.0,58.693724,109.498371,0.0,19.0,155.0,524.14,1339.0,39.7
calculated_host_listings_count,2932.0,12.156207,28.026407,1.0,2.0,33.0,129.00,129.0,10.2


### 4. Initial investigation of the listings data

**Steps taken**

- Checked shape, column types and non-null counts with .shape, .info() and .describe().
- Listed text columns and previewed price, first_review and last_review.
- Counted missing values per column, as counts and percentages.
- Identified columns with only one distinct value.
- Checked for duplicate rows and unique listing IDs.
- Looked at distributions of the key numeric columns (mean against median, extremes).

**Structure and data types**

- 2,932 listings and 90 columns: 38 float, 24 integer, 28 text.
- Several text columns should be other types: price (stored with a currency symbol), first_review and last_review (dates), and true/false columns such as host_is_superhost.
- id, host_id and host_profile_id are identifiers, not measurements. Their summary statistics are meaningless and will be excluded from analysis.

**Missing values**

- 15 columns are 100% empty (including license, bathrooms, instant_bookable and the host_response_* fields).
- Price and its derived fields (price_quote_*, estimated_revenue_l365d) are missing for 671 listings (22.9%). The cause is investigated in the missing-values step of the transformation stage.
- Review-score columns and reviews_per_month are missing for 288 listings (9.8%). These are listings with no reviews (confirmed in the transformation stage).

**Constant and duplicate checks**

- 18 columns carry a single value (15 entirely empty, plus scrape_id, last_scraped and calendar_last_scraped).
- There are no duplicate rows and every id is unique.

**Distribution**
- Nearly three-quarters of listings are entire homes. Those are the ones that take a home off the long-term rental market, so this is the most relevant figure for "short-let pressure".
- Hotel and shared rooms are about 6 listings each, too few to analyse on their own. They can be grouped together with private rooms.

**Observations to follow up**

- Some listings appear dormant (no price, last review in 2020), which could inflate supply counts. A definition of an "active" listing is needed before aggregating.
- The $ in price raises a currency question. BUT the data dictionary confirms "the $ sign is a technical artifact of the export" and can be ignored.
host_listings_count (mean 29.4, max 1,217) differs sharply from calculated_host_listings_count (mean 12.2, max 129), suggesting different scopes. This is not yet confirmed: the first rows I compared matched calculated_host_listings_count. 

**Planned for the transformation stage**

- Flag dormant listings without deleting them.
- Drop constant, empty and personal-data columns, recording a reason for each.
- Convert price, date and true/false columns, and decide how to treat missing values.
- Define "operator listing" explicitly and create the flags.


---

# Transforming the data #

**Plan and rationale**

**Purpose:**

Turn the raw Inside Airbnb listings into a table fit for analysis, with every change recorded and justified. Raw files in data/raw/ stay untouched. All work happens on a copy, and the result is saved to data/processed/.

**Principle:** 

Each step records what changed (row and column counts before and after) and why. Where a choice affects the findings, the effect is tested, not assumed.

**Steps and reasoning:**

- Copy the data. Keeps the original recoverable and gives a baseline to reconcile against at the end.
- Flag dormant listings (is_active), do not delete them. Dormant listings may inflate supply counts, but "dormant" is a judgement built on two proxies. A flag keeps the decision reversible, lets key results be compared with and without them, and keeps the judgement visible.
- Drop columns that cannot support the analysis. Empty, constant, personal-data and duplicate columns, each with a recorded reason. Personal data must not appear in a public repository.
- Fix data types. Prices, dates and true/false columns are stored as text, so calculations and summaries on them would be wrong. Identifiers become labels.
- Handle missing values. Each column is classified before it is treated: missing by design (no reviews, so no score), missing without a known cause (price), or unreliable (has_availability). Averages are not used to fill gaps, because the gaps are not random.
- Check outliers. Skewed columns can be dominated by a few extreme values. Each extreme is judged as an error, a genuine extreme or a case to cap, and the reasoning is recorded.
- Create features. The operator flag (3 or more listings, with 2, 5 and 10 as sensitivity variants), a whole-home flag and a short-stay flag. These are the measures the hypotheses rely on.
- Validate and save. Re-run the missing-value, type and duplicate checks, confirm all 34 neighbourhoods still have listings, reconcile the row and column counts, and save to data/processed/.



## 1. Create a working copy ##

**1. Work on a copy and record the starting point**
- Start from 2,932 rows and 90 columns.
- *Why:* keeps the original recoverable and gives a baseline to reconcile against at the end.

In [26]:
df = listings.copy()

start_rows, start_cols = df.shape
print(f"Starting point: {start_rows} rows x {start_cols} columns")

Starting point: 2932 rows x 90 columns


In [27]:
print("Same shape as original:", df.shape == listings.shape)
print("Same contents:", df.equals(listings))
print("Same object in memory:", df is listings)  

Same shape as original: True
Same contents: True
Same object in memory: False


## 2. Flag dormant listings ##

**Flag dormant listings (`is_active`), without deleting them**
- Rule: a listing is **inactive** if `availability_365` is 0 **and** `number_of_reviews_ltm` is 0 (zero availability and no reviews in the last 12 months). Our analysis confirms that 371 are inactive and 2,561 active.
- *Why a combined rule:* zero availability alone would wrongly flag 220 listings that were reviewed in the last 12 months and look active with a blocked calendar (or are fully booked). Requiring no recent reviews as well targets listings that look genuinely inactive.
- *Why flag instead of delete:*
  - **Reversible.** The decision can be changed without reloading and re-cleaning.
  - **Testable.** Key results can be compared with and without inactive listings. This matters because removing them changes the findings (the share of listings run by multi-listing hosts moves from roughly 38.5% to 42.0%).
  - **Defensible.** The 12-month cut-off and the two proxies are judgement calls, not ground truth. A flag keeps that judgement visible and open to challenge.
  - **Honest about what is dropped.** Inactive listings still say something about past supply, so deleting them would hide that.
- *Default:* All-listings results are reported as a robustness check.
- *Known limits:* the rule is conservative, and some listings with availability but no recent reviews may also be dormant. Very small areas shrink further (the smallest falls from 17 to 14 listings).

In [28]:
df["is_active"] = ~((df["availability_365"] == 0) & (df["number_of_reviews_ltm"] == 0))

print(df["is_active"].value_counts())
print("Rows unchanged:", len(df) == start_rows)

is_active
True     2561
False     371
Name: count, dtype: int64
Rows unchanged: True


In [29]:

ct = pd.crosstab(df["availability_365"] == 0, df["number_of_reviews_ltm"] == 0)
ct = ct.rename(index={False: "Has availability", True: "Zero availability"},
               columns={False: "Reviews in last 12 months", True: "No reviews in last 12 months"})
ct.index.name = None
ct.columns.name = None
ct


,Reviews in last 12 months,No reviews in last 12 months
Has availability,1870,471
Zero availability,220,371


In [30]:
for label, data in [("All listings", df), ("Active only", df[df["is_active"]])]:
    print(
        label,
        "| listings:", len(data),
        "| entire homes:", round((data["room_type"] == "Entire home/apt").mean(), 3),
        "| 3+ listing hosts:", round((data["calculated_host_listings_count"] >= 3).mean(), 3),
    )

All listings | listings: 2932 | entire homes: 0.721 | 3+ listing hosts: 0.385
Active only | listings: 2561 | entire homes: 0.73 | 3+ listing hosts: 0.42


**Result of step 2**

- 371 listings (12.7%) are flagged inactive, 2,561 active. The row count is unchanged, because nothing was deleted.
- The crosstab confirms the combined rule is stricter than zero availability alone: 220 listings have zero availability but were reviewed in the last 12 months, so they stay active.
- The comparison confirms that the share of listings with a host who has 3 or more listings moves from about 38.5% (all listings) to about 42.0% (active only). 
- Deleting the dormant listings would therefore have changed the headline result, which is why the flag is kept and the effect tested.
- Later finding (step 5): all 371 inactive listings also come from the previous scrape, not the latest one. The flag and the scrape source overlap, and the analysis has to keep them apart.



## 3. Drop columns that cannot support the analysis ##

The dataset contains a lot of colums that are either empty or that carry no information. There also several that contain information that is not relevant to our analysis. Dropping the rest keeps the analysis focused on the question.

Each drop is listed with a one-line reason in drop_log, saved to docs/dropped_columns.csv. This change log idea is a result of using Claude to improve the way this project. I would not have thought of it on my own and it provides us a single resource showing exactly what was dropped and why at the end of the process. This will then be used in the READ.ME.

- Constant or entirely empty columns (18, for example license, bathrooms, scrape_id). They carry no information.
- Personal data, URLs and free text (host names, profile and picture links, host_about, host_location, name, description). They are not needed, and personal data must not appear in a public repository (an assessment requirement).
- Host identifiers. host_id is replaced by an anonymous host_key created with pd.factorize, so the same host still gets the same number but Airbnb's ID is not published. host_profile_id is redundant.
- Duplicates of other columns. price_quote_price_per_night is checked against price before dropping. The other price_quote_* columns describe one sample stay, not a lasting listing attribute, so they are dropped for that reason (not because they duplicate price).
- Columns not relevant to the question. host_has_profile_pic and host_identity_verified (hosts' profile details) are not needed for operator concentration or listing behaviour.
- The six minimum/maximum-nights variants and host_listings_count are dropped as they do not add to our analysis.

Kept deliberately: 

- latitude and longitude (for the dashboard map) 
- neighbourhood_cleansed (the area key) 
- NOTE: The GeoJSON joins to the listings on neighbourhood name, so replacing host_id with host_key does not affect any map.

In [31]:
drop_log = {}

def drop_columns(frame, cols, reason):
    """Drop columns that exist, log the reason for each, and report the new shape."""
    not_found = [c for c in cols if c not in frame.columns]
    if not_found:
        print("Not found (already dropped or misspelt):", not_found)
    cols = [c for c in cols if c in frame.columns]
    for c in cols:
        drop_log[c] = reason
    out = frame.drop(columns=cols)
    print(f"Dropped {len(cols)} columns -> {out.shape}")
    return out

In [32]:
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print(len(constant_cols), constant_cols)   

df = drop_columns(df, constant_cols, "Constant or entirely empty: carries no information")

18 ['scrape_id', 'last_scraped', 'neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated', 'calendar_last_scraped', 'license', 'instant_bookable']
Dropped 18 columns -> (2932, 73)


We will drop the two columns that list how long the hosts have been an airbnb user. But we will keep the columns that say how long an individual as been a host - as these columns are a potentially important way of seperating professional operators and more casual hosts.

In [33]:
df = drop_columns(
    df,
    ["hosts_time_as_user_months", "hosts_time_as_user_years"],
    "Host tenure: not useful to the analysis",
)

Dropped 2 columns -> (2932, 71)


In [34]:
personal_cols = [
    "listing_url", "picture_url", "host_url", "host_profile_url", "host_picture_url",
    "host_name", "host_about", "host_location", "name", "description",
]
df = drop_columns(df, personal_cols, "Personal data, URL or free text: not needed and shouldn't be published")

Dropped 10 columns -> (2932, 61)


In [35]:
df = drop_columns(df, ["host_has_profile_pic", "host_identity_verified"],
                  "Host profile details: not relevant to operator concentration or listing behaviour")

Dropped 2 columns -> (2932, 59)


We will also replace host_id with a newly created host_key. This will not impact our ability to use the GeoJSON.

In [36]:
df["host_key"] = pd.factorize(df["host_id"])[0]


print("Distinct hosts match:", df["host_key"].nunique() == df["host_id"].nunique())

Distinct hosts match: True


In [37]:
df = drop_columns(
    df,
    ["host_id", "host_profile_id"],
    "Replaced by anonymous host_key (host_id); host_profile_id is redundant",
)

Dropped 2 columns -> (2932, 58)


In [38]:

per_host = df.groupby("host_key")["id"].transform("size")
print("Match:", (per_host == df["calculated_host_listings_count"]).mean())

Match: 1.0


In [39]:
print("host_key" in df.columns)

True


In [40]:
# Proof first: is the per-night quote the same as price?
price_num = pd.to_numeric(listings["price"].str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
diff = (price_num - listings["price_quote_price_per_night"]).abs()
print("Max difference:", diff.max(), "| rows compared:", diff.notna().sum())

Max difference: 0.0 | rows compared: 2261


In [41]:
# Are the six nights variants copies of the two main columns?
min_like = ["minimum_minimum_nights", "maximum_minimum_nights", "minimum_nights_avg_ntm"]
max_like = ["minimum_maximum_nights", "maximum_maximum_nights", "maximum_nights_avg_ntm"]
for c in min_like:
    print(c, "equals minimum_nights:", (listings[c] == listings["minimum_nights"]).mean().round(3))
for c in max_like:
    print(c, "equals maximum_nights:", (listings[c] == listings["maximum_nights"]).mean().round(3))

# Is host_listings_count the same as the calculated count?
print("host_listings_count matches calculated:",
      (listings["host_listings_count"] == listings["calculated_host_listings_count"]).mean().round(3))

minimum_minimum_nights equals minimum_nights: 1.0
maximum_minimum_nights equals minimum_nights: 0.642
minimum_nights_avg_ntm equals minimum_nights: 0.661
minimum_maximum_nights equals maximum_nights: 0.959
maximum_maximum_nights equals maximum_nights: 1.0
maximum_nights_avg_ntm equals maximum_nights: 0.959
host_listings_count matches calculated: 0.657


In [53]:
diff = listings["host_listings_count"] - listings["calculated_host_listings_count"]
print((diff >= 0).mean().round(3))      # share where the broader count is at least as large
print(diff.describe())

1.0
count    2932.000000
mean       17.290587
std        84.598577
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max      1215.000000
dtype: float64


**Nights variants and `host_listings_count`: why they were dropped**

*Nights variants (six columns dropped).* Inside Airbnb provides several versions of the minimum- and maximum-stay fields alongside `minimum_nights` and `maximum_nights`. These appear to summarise the stay limits across the calendar window. I compared each with its main column before deciding:


- Two columns are exact duplicates in this snapshot, so they add nothing.
- The other four are not duplicates. They differ for between 4% and 36% of listings, so they carry some different information.
- They are still dropped, because they are not needed to test H1 to H3. H2 compares minimum stay between operator and casual listings, and `minimum_nights` measures that directly. Keeping four extra variants of the same idea would add columns to explain without answering a question.
- `minimum_nights` and `maximum_nights` are kept.
- *Limitation:* if minimum stay varies through the year for some listings, `minimum_nights` will not show it. Using `minimum_nights_avg_ntm` is a possible extension.

*`host_listings_count` (one column dropped).* This column is not a duplicate of `calculated_host_listings_count`: they match for only 65.7% of listings.
- `host_listings_count` is never smaller than the calculated count (100% of rows), and the gap is large for a minority of hosts (maximum 1,215 against 129 in the calculated count). This suggests it counts a host's listings beyond this Bristol snapshot. This is supported by the data dictionary which states that it counts "The number of listings the host has" - but not confirmed.
- The operator definition for this project is based on the number of a host's listings **in this dataset**, so `calculated_host_listings_count` is used throughout. Using two different counts would make the operator flag inconsistent.
- It is also a host-count column, so it would have to be excluded from the classifier's inputs anyway, to avoid circularity.
- *Limitation:* a host with one Bristol listing and a large portfolio elsewhere is treated as a casual host here. A wider operator definition using this column is a possible extension.

In [ ]:
df = drop_columns(df, ["minimum_minimum_nights", "maximum_minimum_nights", "minimum_nights_avg_ntm",
                       "minimum_maximum_nights", "maximum_maximum_nights", "maximum_nights_avg_ntm"],
                  "Calendar-window variants of minimum/maximum nights: not needed for H1 to H3; minimum_nights and maximum_nights kept")
df = drop_columns(df, ["host_listings_count"],
                  "Different scope from the Bristol count; analysis uses calculated_host_listings_count throughout")



In [42]:
df = drop_columns(df, ["price_quote_price_per_night"],
                  "Duplicate of price (max difference checked)")
df = drop_columns(df, ["price_quote_total_price", "price_quote_checkin_date",
                       "price_quote_checkout_date", "price_quote_raw"],
                  "Quote for one sample stay, not a stable listing attribute; nightly price is in price")

Dropped 1 columns -> (2932, 57)
Dropped 4 columns -> (2932, 53)


**Result of step 3**

| Step | What changed | Evidence | Why |
| --- | --- | --- | --- |
| 1. Copy | Worked on a copy: 2,932 rows x 90 columns | Shape and contents matched the original; not the same object in memory | Keeps the raw data recoverable |
| 2. Dormant flag | `is_active` added: 371 inactive, 2,561 active. No rows deleted. | Without deleting, the 3+ listing host share moves from 38.5% (all) to 42.0% (active only) | Dormancy is a judgement, so it stays visible and testable |
| 3. Drop columns | [RECORD: 46] columns dropped, each with a logged reason (`docs/dropped_columns.csv`). Final shape [RECORD: 2,932 x 46]. | Log reconciles with the dropped columns: [RECORD: True]. Price proof: maximum difference 0.0 over 2,261 rows. | Empty, constant, personal-data, duplicate and out-of-scope columns add nothing or must not be published |
| 4. Data types | `price` numeric, review dates as dates, `host_is_superhost` boolean, `id` as text. `has_availability` dropped. | 585 of 2,914 listings flagged true (20.1%) have zero available days | Calculations on text are wrong. `has_availability` contradicts the calendar data. |
| 5. Missing values | Left as missing, with flags. Nothing imputed. | See below | The gaps are not random, so filling them would invent data |

**Missing values**
- Review scores and dates (288, 9.8%): all are listings with zero reviews, so there is no value to record. Left as missing.
- Price and `estimated_revenue_l365d` (671, 22.9%): left as missing with a `price_missing` flag. Dropping would lose nearly a quarter of the data. [CONFIRM: most are zero-availability or previous-scrape listings.]
- Every column other than price and reviews has fewer than [RECORD] missing values and is not used in H1 to H3.

**Decisions that affect later results**
- Dormant listings are kept and flagged. Results are reported for all listings and for active listings only.
- `host_key` replaces `host_id`. Hosts are pseudonymised, not anonymised: listing IDs and the raw files still identify them.
- The operator definition uses `calculated_host_listings_count` (Bristol only). `host_listings_count` has a different scope and was dropped.

**Limitations**
- Whether a listing is dormant rests on two proxies, not ground truth.
- Price is missing for 22.9% of listings, so any price-based result describes listings that have a price.
- [Add: the scrape-source finding once the code evidencing it is in the notebook.]

In [44]:
dropped = set(listings.columns) - set(df.columns)
print("Log matches dropped columns:", dropped == set(drop_log))
print("Differences:", dropped ^ set(drop_log)) 

Log matches dropped columns: True
Differences: set()


In [45]:
drop_table = (
    pd.DataFrame(drop_log.items(), columns=["column", "reason"])
    .sort_values(["reason", "column"])
    .reset_index(drop=True)
)
drop_table

,column,reason
0,bathrooms,Constant or entirely empty: carries no informa...
1,calendar_last_scraped,Constant or entirely empty: carries no informa...
2,calendar_updated,Constant or entirely empty: carries no informa...
3,host_acceptance_rate,Constant or entirely empty: carries no informa...
4,host_neighbourhood,Constant or entirely empty: carries no informa...
5,host_response_rate,Constant or entirely empty: carries no informa...
6,host_response_time,Constant or entirely empty: carries no informa...
7,host_since,Constant or entirely empty: carries no informa...
8,host_thumbnail_url,Constant or entirely empty: carries no informa...
9,host_total_listings_count,Constant or entirely empty: carries no informa...


In [46]:
drop_table.groupby("reason")["column"].count()

reason
Constant or entirely empty: carries no information                                      18
Duplicate of price (max difference checked)                                              1
Host profile details: not relevant to operator concentration or listing behaviour        2
Host tenure: not useful to the analysis                                                  2
Personal data, URL or free text: not needed and shouldn't be published                  10
Quote for one sample stay, not a stable listing attribute; nightly price is in price     4
Replaced by anonymous host_key (host_id); host_profile_id is redundant                   2
Name: column, dtype: int64

In [ ]:
DOCS = Path("docs")
DOCS.mkdir(exist_ok=True)
drop_table.to_csv(DOCS / "dropped_columns.csv", index=False)
print((DOCS / "dropped_columns.csv").resolve())

C:\Users\chheb\OneDrive\Documents\Programming\docs\dropped_columns.csv


## 4. Converting data types ##

**4. Fix data types**
- `price`: remove the currency symbol and convert to a number. [Confirm currency in the data dictionary.]
- `first_review` and `last_review`: convert to dates.
- True/false text columns: convert to booleans.
- Identifier columns (`id`, `host_id`): keep as labels, not numbers.
- *Why:* summaries and calculations on text or identifier columns are wrong or meaningless (for example, a mean of listing IDs).

In [48]:
dtypes_before = df.dtypes.copy()

In [49]:
prices = df["price"].dropna()
print("All match $ pattern:", prices.str.match(r"^\$[\d,]+(\.\d+)?$").all())

All match $ pattern: True


In [50]:
df["price"] = pd.to_numeric(df["price"].str.replace(r"[$,]", "", regex=True))
print(df["price"].describe())

count    2261.000000
mean      157.217355
std       141.053495
min        24.600000
25%        83.000000
50%       130.000000
75%       189.500000
max      2238.000000
Name: price, dtype: float64


In [51]:
before = df[["first_review", "last_review"]].isnull().sum()
df["first_review"] = pd.to_datetime(df["first_review"], errors="coerce")
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")
after = df[["first_review", "last_review"]].isnull().sum()
print(before.equals(after))

True


In [52]:
tf_cols = ["host_is_superhost", "has_availability", "host_identity_verified", "host_has_profile_pic"]
for c in tf_cols:
    print(c, df[c].value_counts(dropna=False).to_dict())

host_is_superhost {'f': 1919, 't': 1013}
has_availability {'t': 2914, nan: 18}


KeyError: 'host_identity_verified'

### Identifying issue with has_availiability ###

In [ ]:
check = pd.crosstab(
    df["has_availability"].astype("object").fillna("blank"),
    df["availability_365"] == 0,
)
check.columns = ["Has available days", "Zero available days"]
check.index.name = "has_availability"
check

,Has available days,Zero available days
has_availability,,
blank,12,6
t,2329,585


In [ ]:
print(df["has_availability"].dtype)          # object = still text
print(df["has_availability"].unique())       # expect 't', 'f' or nan

df["has_availability"] = df["has_availability"].map({"t": True, "f": False}).astype("boolean")

str
<ArrowStringArray>
['t', nan]
Length: 2, dtype: str


In [ ]:
flagged_true = df["has_availability"].fillna(False)
contradictions = flagged_true & (df["availability_365"] == 0)
print(f"{contradictions.sum()} of {flagged_true.sum()} listings flagged True "
      f"({contradictions.sum() / flagged_true.sum():.1%}) have zero available days")

585 of 2914 listings flagged True (20.1%) have zero available days


In [ ]:
df = drop_columns(df, ["has_availability"],
                  "Unreliable: flagged True for 585 listings whose calendar shows zero available days")

Dropped 1 columns -> (2932, 58)


In [ ]:
df.loc[contradictions, ["id", "has_availability", "availability_30", "availability_365"]].head()

,id,has_availability,availability_30,availability_365
3,390781,True,0,0
8,868888,True,0,0
10,975170,True,0,0
16,1457693,True,0,0
17,1536968,True,0,0


In [ ]:
for c in tf_cols:
    df[c] = df[c].map({"t": True, "f": False}).astype("boolean")

In [ ]:
df["id"] = df["id"].astype(str)

In [ ]:
types_changed = pd.DataFrame({"before": dtypes_before, "after": df.dtypes})
types_changed[types_changed["before"].astype(str) != types_changed["after"].astype(str)]

,before,after
id,int64,str
host_is_superhost,str,boolean
host_has_profile_pic,str,boolean
host_identity_verified,str,boolean
price,str,float64
has_availability,str,boolean
first_review,str,datetime64[us]
last_review,str,datetime64[us]


In [ ]:
df.head()

,id,source,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_is_superhost,host_listings_count,host_has_profile_pic,host_identity_verified,...,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,is_active,host_key
0,70820,NaN,15,8,15,7,False,5,True,True,...,4.93,4.76,4.76,5,0,5,0,1.36,True,0
1,176306,NaN,15,3,15,3,False,1,True,True,...,4.90,4.78,4.69,1,0,1,0,0.36,True,1
2,307634,NaN,14,8,9,5,True,3,True,True,...,4.84,4.86,4.74,3,0,3,0,0.47,True,2
3,390781,previous scrape,14,6,14,5,False,2,True,True,...,4.93,4.72,4.84,2,0,2,0,0.34,True,3
4,407485,NaN,14,5,14,5,False,1,True,True,...,4.92,4.79,4.70,1,1,0,0,2.35,True,4


**5. Handle missing values**
- Missing prices: recount after the active flag, then decide how to treat the remainder. Most missing prices belong to listings with zero availability.
- Missing review scores: leave blank, since a listing with no reviews has no score.
- *Why:* the missingness is not random, so filling with an average could invent data for listings that are not for sale.

In [ ]:
missing = (
    df.isna().sum()
      .to_frame("n_missing")
      .assign(pct=lambda t: (t["n_missing"] / len(df) * 100).round(1))
      .query("n_missing > 0")
      .sort_values("n_missing", ascending=False)
      .reset_index(names="column")
)
missing

,column,n_missing,pct
0,source,2227,76.0
1,price,671,22.9
2,price_quote_price_per_night,671,22.9
3,price_quote_total_price,671,22.9
4,estimated_revenue_l365d,671,22.9
5,price_quote_raw,616,21.0
6,price_quote_checkin_date,616,21.0
7,price_quote_checkout_date,616,21.0
8,review_scores_communication,288,9.8
9,first_review,288,9.8


In [ ]:
# Do missing review scores coincide with zero reviews?
print(pd.crosstab(df["review_scores_rating"].isna(), df["number_of_reviews"] == 0))

# Do the 81 missing prices look different (active? operators?)
print(df[df["price"].isna()]["is_active"].value_counts())

number_of_reviews     False  True 
review_scores_rating              
False                  2644      0
True                      0    288
is_active
False    371
True     300
Name: count, dtype: int64


In [ ]:
df["price_missing"] = df["price"].isna()

missing_log = {}
missing_log["review_scores_*"] = ("288 (9.8%)", "left as NaN",
    "structurally missing: all coincide with zero reviews; H3 uses the 2,644 reviewed listings")
missing_log["price"] = ("671 (22.9%)", "left as NaN + price_missing flag",
    "too large to drop or impute; not needed for H1-H3; missingness itself to be examined")

print(df["price_missing"].sum())

671


In [ ]:
df

,id,source,hosts_time_as_host_years,hosts_time_as_host_months,host_is_superhost,host_listings_count,neighbourhood_cleansed,latitude,longitude,property_type,...,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,is_active,host_key
0,70820,NaN,15,7,f,5,Windmill Hill,51.438430,-2.591830,Private room in townhouse,...,4.93,4.76,4.76,5,0,5,0,1.36,True,0
1,176306,NaN,15,3,f,1,Southville,51.441740,-2.602970,Private room in townhouse,...,4.90,4.78,4.69,1,0,1,0,0.36,True,1
2,307634,NaN,9,5,t,3,Easton,51.464870,-2.556900,Private room in townhouse,...,4.84,4.86,4.74,3,0,3,0,0.47,True,2
3,390781,previous scrape,14,5,f,2,Brislington West,51.444440,-2.557830,Private room in home,...,4.93,4.72,4.84,2,0,2,0,0.34,True,3
4,407485,NaN,14,5,f,1,Ashley,51.467690,-2.582500,Entire rental unit,...,4.92,4.79,4.70,1,1,0,0,2.35,True,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927,1777398278259778680,NaN,4,5,f,34,Easton,51.462059,-2.557122,Entire rental unit,...,NaN,NaN,NaN,25,25,0,0,NaN,True,534
2928,1777545990995702017,NaN,6,8,t,3,Redland,51.478170,-2.606940,Private room in rental unit,...,NaN,NaN,NaN,3,2,1,0,NaN,True,1344
2929,1779173612058982519,NaN,7,1,f,1,Filwood,51.433141,-2.589852,Private room in bed and breakfast,...,NaN,NaN,NaN,1,0,1,0,NaN,True,1751
2930,1779568129128286369,NaN,12,11,f,1,Ashley,51.469630,-2.576680,Private room in home,...,NaN,NaN,NaN,1,0,1,0,NaN,True,1752


**6. Check outliers**
- Review extremes in price, `minimum_nights`, `accommodates` and `calculated_host_listings_count`.
- Examine the largest host (129 listings) individually.
- *Why:* in skewed data like this, a few large values can dominate averages. Each one is treated as an error, a genuine extreme, or a case to cap, and the reasoning is recorded.

**7. Create new features**
- Operator flag (host with 3 or more listings), with 2, 5 and 10 as variants for sensitivity.
- Whole-home flag and short-stay flag.
- *Why:* these are the measures the hypotheses rely on. The operator threshold is a judgement, so alternatives are tested.

**8. Validate and save**
- Re-run missing-value, type and duplicate checks.
- Confirm all 34 neighbourhoods still have listings.
- Reconcile row and column counts from start to finish.
- Save to `data/processed/` and note the file name.
- *Why:* proves the transformations did what was intended and leaves the pipeline reproducible.

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)